# 07 — Predictive model: the final figures

The manuscript's **Figure 4** and **Figure 5**, and nothing else. The exploration behind them
(multiverses, rankings, modelling choices, volcanos, SHAP shares, prediction checks) stays in
`06_PredictiveModel_Exploration` (notebooks 10 + 13).

* **Figure 4**, two halves side by side (composed in the last cell). Left (was plot 10.9): Cam-CAN,
  cross-validated (train on 80%, test on 20%): brain-state transition model (bstm) beside static FC
  (fcm, Yeo 7 by default), anxiety and depression, rest and movie, over the Age + Sex baseline, with
  what the bstm model uses below. Right (was plot 13.4): the same, trained on all of Cam-CAN and
  tested once on HCP.
* **Figure 5** (was plot 10.12), Cam-CAN. Where the model looks: the state graphs per group and condition, red
  = the model's top variables that also differ from the Normal group.

Section 1 (Cam-CAN) and section 2 (transfer to HCP) each have their own setup cell, as in 10 and 13.


## 1. Cam-CAN: setup, state set and model

In [ ]:
import sys
from pathlib import Path

# THE PACKAGE, which is not pip-installed inside the analysis image. A Jupyter
# kernel starts with sys.path[0] = the notebook's own directory, so without
# this `import fmri_decomposition` fails from notebooks/.
#
# The REPO ROOT and nothing else. `notebooks/` used to be on this list too,
# back when the table and figure modules lived beside the .ipynb files; that
# part is gone and must not come back -- a directory on sys.path can shadow a
# real package, which is how an ImportError turns into the wrong module.
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from fmri_decomposition.supervised_model import hads as ht
from fmri_decomposition.figures import hads as hp

cfg = ht.Settings(
    # the statistical rule (every volcano, every diagram)
    case_from="Mild",               # this HADS level and above = the phenotype group
    alpha=0.10,                     # BH-FDR
    fc_line=0.25,                   # |log2 fold change| needed too; None = FDR only
    shrink=1.0,                     # log2fc pseudocount: 0 = raw ratio (widest), 5+ = narrow
    min_present=0.10,               # drop cells / occupancies non-zero in < this share of subjects
    covariates=["Age", "Sex", "n_transitions"],
    entropy_estimator="miller_madow",   # entropy_rate_bits recomputed with the bias correction;
                                        # None = as stored in the tables (plug-in)
    # A: the multiverses
    multiverse_arm="all [cond]",    # its variables: cells as p(j | i), occupancy, dynamics
    xlim=2,                         # same x range ±xlim in every volcano; None = fit each
    label_max=25,                   # names per volcano panel
    # B: performance
    models=None,                    # None = every model but logistic
    arms=None,                      # None = every arm, e.g. ["cells [joint]", "edges"]
    baseline_covariates=["Age", "Sex"],
    seeds=[0, 1, 2, 3, 4],          # as select-bstm / select-fcm
    margin_sd=0,                    # "above" = mean - margin_sd * row sd > baseline + its sd
    highlight_max=10,               # rows named per B1 panel; the table has all
    # "K_" is the number of states. It belongs here because it is the one axis
    # that changes HOW MANY PREDICTORS the model must estimate: K states give
    # K*K transition cells, so K=27 is 729 of them and K=8 is 64, from the
    # same subjects. B2.K (fig_k_curve) reads that axis further.
    factors={"window": "window size", "atlas": "atlas", "model": "model",
             "method": "state method", "arm": "arm",
             "K_": "number of states (K)"},
    factor_trees=["bstm"],          # add "fcm" to pool its rows where the axis exists
    # C: SHAP shares
    share_arms=["all [joint]", "all [cond]"],
    share_top=10,                   # best state sets per (model, arm); None = all (hours of CPU)
    shap_seed=0,
    # D / E
    graph_arms=["all [joint]", "all [cond]"],   # the arms the default choices are drawn from
    edge_top=12,                    # the Normal group's strongest arrows per task
    label_pp=None,                  # None = every highlight shows its pp; e.g. 12 = the largest only
    top_vars=15,                    # E2.2: variables per panel
    seeds_stab=[0, 1, 2, 3, 4],     # E2.2 / E3: fold seeds
    highlight_top=8,                # E3: the model's top graph variables by mean |SHAP|
    stable_min=0.8,                 # E3: ... in the top for this share of the seeds
    min_d=0.2,                      # E3: ... and |Cohen's d| vs Normal at least this
    # every figure
    font_scale=2.0,                 # every text x this (1 = matplotlib's sizes)
    fig_scale=1.5,                  # every figure's size x this, to make room for the text
    dyn_gap=2.0,                    # D3 / E3: space between the dynamics items
    dyn_away=4.0,                   # D3 / E3: space between each diagram and its dynamics
    dyn_icon=1.0,                   # D3 / E3: size of its symbols, x the font
    save_dir=None,                  # e.g. "figures/10" to save PNG + SVG
)
st = ht.Study(cfg)
T0, T1 = cfg.tasks                  # the base task (rest) and the other (movie)
print(f"phenotype: {len(st.pheno):,} subjects;",
      ", ".join(f"{lab} {st.pheno[lab].value_counts().sort_index().astype(int).to_dict()}" for lab in cfg.targets))
# Check this before reading a gap as a result: the runs are separate, so `n` can differ.
st.summ.pivot_table(index=["tree", "task"], columns="phenotype", values="n", aggfunc="max")

In [ ]:
base = st.baselines()
options = st.space_options(base)                 # the best state sets (no multiverse needed here)
for _, r in options.iterrows():
    print(f"SPACE = {ht.as_choice(r, ['atlas', 'window_s', 'states'])}   # {r['model']} · {r['arm']}  ρ={r['cv']:.3f}")
SPACE = None                        # e.g. dict(atlas="harvardoxford", window_s=30.0, states="HMM2_umap3_8")
space = st.choose_space(SPACE, options)
print("\nspace:", ht.set_label(space))


In [ ]:
model_options = st.model_options(space)
for _, r in model_options.iterrows():
    print(f"MODEL = {ht.as_choice(r, ['model', 'arm'])}   # ρ={r['cv']:.3f}")
MODEL = None                        # e.g. dict(model="hgb", arm="all [cond]")
model = st.choose_model(MODEL, model_options)
label = f"{model['model']} · {model['arm']} · {ht.set_label(space)}"
print("\nmodel:", label)
# model_options

## Figure 4, left half — cross-validated in Cam-CAN (was plot 10.9)

Drawn here, shown with the right half in the Figure 4 cell at the end.


In [ ]:
E1_MODEL = "hgb"                    # or "ridge"
FC_ARM = "edges"                    # or "global" (each person's mean and SD over the edges)
FC_ATLAS = "yeo7"                   # the static-FC comparison: "yeo7" (main figure, matched in
                                    # dimensionality) | None = the states' atlas (Extended Data)
perf = st.performance_trees(space, model["arm"], E1_MODEL, base, fc_arm=FC_ARM, fc_atlas=FC_ATLAS)
e1_seeds = st.tree_seeds(space, model["arm"], E1_MODEL, fc_arm=FC_ARM, fc_atlas=FC_ATLAS)
e1_shares = st.block_shares(st.deep(space, dict(model=E1_MODEL, arm=model["arm"])))
_fc_atlas = FC_ATLAS or space["atlas"]
fc_note = "" if (perf["tree"] == "fcm").any() else f" -- no fcm run on {_fc_atlas}"
fig4_left = hp.fig_performance(perf, cfg, "",
                   {"bstm": f"selected: {ht.set_label(space)} · {model['arm']}",
                    "fcm": f"{_fc_atlas} · same model · {FC_ARM}{fc_note}"},
                   E1_MODEL, seeds=e1_seeds, shares=e1_shares, show=False)


## Figure 5 — where the model looks, Cam-CAN (was plot 10.12)

The statistics behind the red: the group tests (D1, D2 of notebook 10, computed here without their volcanos), the pooled graphs and the model's out-of-fold SHAP.

In [ ]:
paired = st.paired(space)              # movie vs rest, Normal group (for the dynamics beside the graphs)
grp = st.group_tests(space)            # each phenotype group vs Normal, adjusted
pooled = st.pooled(space)
dyn = st.dynamics_marks(space, paired, grp)
deep = st.deep(space, model)
st.check(deep)


In [ ]:
mmarks = st.model_marks(space, deep, pooled)
refs_model = {(t, lab): (t, "Normal") for t in cfg.tasks for lab in cfg.targets}
hp.fig_graph(pooled, mmarks, cfg, refs_model, {k: hp.RED for k in refs_model}, f"Figure 5  where the model looks · {label}",
             [(hp.RED, f"red = the model's top {cfg.highlight_top} graph variables by mean |SHAP|, top in "
                       f"≥{cfg.stable_min:.0%} of seeds, |Cohen's d| ≥ {cfg.min_d} vs Normal; label = change in pp")],
             "figure5_graph", dyn=dyn.assign(hit=False))   # the dynamics as values; red is the model's here
# mmarks.round(3)


## 2. Transfer to HCP: setup and configuration

In [ ]:
import sys
from pathlib import Path

# THE PACKAGE, which is not pip-installed inside the analysis image. A Jupyter
# kernel starts with sys.path[0] = the notebook's own directory, so without
# this `import fmri_decomposition` fails from notebooks/.
#
# The REPO ROOT and nothing else. `notebooks/` used to be on this list too,
# back when the table and figure modules lived beside the .ipynb files; that
# part is gone and must not come back -- a directory on sys.path can shadow a
# real package, which is how an ImportError turns into the wrong module.
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from fmri_decomposition.supervised_model import hads as ht
from fmri_decomposition.supervised_model import transfer_tables as tt
from fmri_decomposition.supervised_model import transfer_config as tcfg
from fmri_decomposition.unsupervised_model import cohort_tables as ct
from fmri_decomposition.figures import hads as hp
from fmri_decomposition.figures import cohorts as cp

# A directory of extra packages installed with `pip --target` -- see
# containers/README.md. APPENDED, never prepended, so nothing in it can shadow
# a package the container was built with; a missing one then fails as a named
# ImportError instead of an ABI segfault.
EXTRA_PYLIBS = None      # e.g. "/project/6008063/tamires/pylibs"
if EXTRA_PYLIBS and EXTRA_PYLIBS not in sys.path:
    sys.path.append(EXTRA_PYLIBS)

SAVE_DIR = None          # e.g. REPO / "figures" / "13"

# HCP's phenotype CSV. The transfer YAML names it as ${HCP_PHENO} and refuses
# to guess when the variable is unset, so the notebook supplies it: the
# environment if exported, else the cluster copy.
import os
HCP_PHENO = os.environ.get(
    "HCP_PHENO", "/project/6008063/tamires/cohorts/hcp_pheno/participants_pheno.csv")

# Which Cam-CAN target set. The rows for BOTH are written by the cluster job;
# this only chooses which to read and draw.
CONFIG = None            # None = config/transfer/camcan_to_hcp.yaml;
                         # a name under config/transfer/, or a path
TARGET = "category"      # "raw" | "category" -- which target set of that config.
                         # "category" is what the manuscript reports (HADS categories,
                         # Methods); the rows live under targets=<this>/ in TRANSFERRED.

# Where `fmri-decomp transfer` wrote them.
TRANSFERRED = None       # None = <outputs>/modelling/version=<v>/
                         #        supervised_model/transferred_models

In [ ]:
# One list, filled by the pooling so the next cell can report it once instead
# of 564 times.
POOLED = []

train, test, train_cfg, test_cfg = tcfg.studies(
    TARGET, config=CONFIG, save_dir=SAVE_DIR, pool_record=POOLED,
    hcp_pheno=HCP_PHENO)
print(f"config   {tcfg.load(CONFIG or tcfg.DEFAULT_CONFIG)['_path']}")
ROOT = (Path(TRANSFERRED) if TRANSFERRED else
        tt.tree_root(train_cfg.root, "transferred_models",
                     train_cfg.modelling_version))

for name, s_ in (("train", train), ("test", test)):
    p = s_.pheno
    print(f"{name:5s}  {len(p):,} subjects;  "
          f"Age {p['Age'].min():.0f}-{p['Age'].max():.0f};  "
          f"Sex {dict(p['Sex'].value_counts())};  "
          + ", ".join(f"{lab} n={int(p[lab].notna().sum())}" for lab in s_.cfg.targets))
print(f"\ntargets  train {dict(train_cfg.targets)}")
print(f"         test  {dict(test_cfg.targets)}")
print(f"rows     {ROOT}")

In [ ]:
raw_rows = tt.read_transferred(ROOT, TARGET, trees=("bstm", "fcm"))
# `drop_degenerate` removes the rows where the model predicted ONE value for
# every test subject -- Spearman has nothing to rank there, so the score is
# NaN. They stay on disk because they are an outcome; it prints which (model,
# arm) they happened to, because "hgb predicted a constant everywhere" is a
# finding about hgb and not a plotting detail.
rows = tt.drop_degenerate(tt.as_report_rows(raw_rows))
prov = raw_rows.iloc[0]
print(f"{len(rows)} configuration(s) over {rows['states'].nunique()} state set(s);  "
      f"{rows['over_base'].mean():.0%} above their transferred baseline")
print(f"  produced by {prov['git_sha']} on {prov['written_at']}, "
      f"seed {prov['seed']}, {prov['n_boot']:,} bootstrap draws, "
      f"grid {prov['grid_hash']}")
if raw_rows["n_imputed_test"].sum():
    print(f"  {int(raw_rows['n_imputed_test'].sum()):,} test value(s) filled with a "
          f"TRAINING column mean (pooling leaves some dynamics undefined); "
          f"{int(raw_rows['n_columns_dropped'].max())} column(s) dropped at most")
# WHAT WAS CONCEDED. A frame mismatch allowed with --allow-frame-mismatch is
# recorded on the row, not just in the job log, because a concession that
# travels only in a log is one nobody reads again.
_m = raw_rows["frames_mismatch"] if "frames_mismatch" in raw_rows else None
if _m is not None and _m.fillna("").ne("").any():
    _bad = raw_rows[_m.fillna("").ne("")]
    print(f"\n!! {len(_bad)} fcm row(s) were scored over DIFFERENT frames:")
    for why, n in _bad["frames_mismatch"].value_counts().items():
        print(f"   {n:>4} row(s)  {why}")
    print("   Allowed on purpose. It has to be stated wherever these rows "
          "appear: in them, a rest-vs-movie or train-vs-test difference is "
          "PARTLY a difference in preprocessing, not in condition or cohort.")

rows.sort_values("score", ascending=False).head(12)[
    ["atlas", "window", "states", "task", "phenotype", "arm", "model",
     "score", "lo", "hi", "base_score"]]

In [ ]:
# ---- which configuration section 3 looks at ------------------------------
# PICK narrows the candidates by any column of `rows`; RANK then takes the
# RANK-th best of what is left. Empty PICK and RANK = 0 is "the best row of
# all", which is the default.
#
#   {"model": "ridge"}                      a model -- and ridge's SHAP needs
#                                           no `shap` package, so E1 draws in
#                                           an image without it
#   {"model": "ridge", "arm": "all"}        a model and an arm
#   {"atlas": "networks", "window_s": 30.0} an atlas and an aperture
#   {"states": "HMM2_pca3_27"}              one state set
#   {"task": "rest", "phenotype": "depression"}   the best cell of one panel
#
# RANK = 1 is the runner-up, 2 the third, and so on -- for checking that the
# answer is not one lucky row.
PICK = {}
RANK = 0

# bstm only: section 3 is about a STATE SET -- its SHAP blocks, its diagram,
# notebook 10's E3 -- and an fcm row has no state set to be about. E1 puts the
# fcm twin beside it either way.
pool = rows[rows["tree"] == "bstm"]
for _col, _want in PICK.items():
    if _col not in pool.columns:
        raise SystemExit(f"{_col!r} is not a column. The ones worth picking on: "
                         f"model, arm, atlas, window_s, states, task, phenotype")
    _left = pool[pool[_col] == _want]
    if not len(_left):
        raise SystemExit(f"nothing left after {_col} == {_want!r}. Present at "
                         f"this point: {sorted(pool[_col].unique())[:12]}")
    pool = _left

clears = pool[pool["lo"] > pool["base_score"]].sort_values("score", ascending=False)
print(f"{len(clears)} of {len(pool)} configuration(s) have an interval clearing "
      f"their baseline")
_ranked = clears if len(clears) else pool.sort_values("score", ascending=False)
if RANK >= len(_ranked):
    raise SystemExit(f"RANK={RANK} but only {len(_ranked)} row(s) to choose from")
best = _ranked.iloc[RANK]

# The ten best, so the next PICK can be read off rather than guessed.
display(_ranked.head(10)[["states", "atlas", "window", "arm", "model", "task",
                          "phenotype", "score", "lo", "hi", "base_score"]])

space = {"atlas": best["atlas"], "window_s": best["window_s"],
         "states": best["states"]}
# task and phenotype BELONG in this line: the score printed is of ONE cell,
# not of the configuration as a whole, and the four cells can disagree.
print(f"\nchosen (rank {RANK}): {best['label']}  ·  {best['arm']}  ·  "
      f"{best['model']}  ·  {best['task']}  ·  {best['phenotype']}")
print(f"  score {best['score']:+.3f}  [{best['lo']:+.3f}, {best['hi']:+.3f}]"
      f"   baseline {best['base_score']:+.3f}")
if not len(clears):
    print("  !! nothing clears its baseline -- this row is the best of a null set")

## Figure 4, right half — trained on all of Cam-CAN, tested on HCP (was plot 13.4)


In [ ]:
FC_ATLAS = "yeo7"   # static-FC comparison: "yeo7" (main figure) | None = the states' atlas (Extended Data)
perf_t = tt.transfer_perf(rows, best, fc_arm=train_cfg.fc_arm, fc_atlas=FC_ATLAS)
display(perf_t[["tree", "arm", "task", "phenotype", "score", "lo", "hi", "base_score", "n_test"]])

# |SHAP| of the Cam-CAN-fitted model, evaluated on the HCP rows it scored
shares_t = tt.transfer_shares(train, test, space, best["arm"], best["model"])

fig4_right = hp.fig_performance(
    perf_t, train_cfg,
    title="",
    legend={"bstm": f"{best['label']} · {best['arm']}",
            "fcm": f"{FC_ATLAS or best['atlas']} · {train_cfg.fc_arm}"},
    model=best["model"], shares=shares_t,
    share_label="share of mean |SHAP| on the TEST subjects "
                "(one fit on all of Cam-CAN, no folds)",
    show=False)


## Figure 4

Left, cross-validated in Cam-CAN (5 folds: train on 80%, test on 20%); right, trained on all of Cam-CAN and tested once on HCP. Each half as drawn above.


In [ ]:
hp.fig_figure4(fig4_left, fig4_right, cfg,
               titles=("Train in 80% Cam-CAN → test in 20% Cam-CAN",
                       "Train in 100% Cam-CAN → test in 100% HCP"));
